# Kolibri-1, hands on: MoE arithmetic, routing load, and German tokenizer math
Companion to the 2026-10-05 post on Aleph Alpha's Kolibri-1 (78.1B total, 3.46B active/token, UniBPE).
All numbers come from the Oct 3-5, 2026 launch coverage and model card.


In [ ]:
TOTAL_B, ACTIVE_B = 78.1, 3.46          # billion params
frac = ACTIVE_B / TOTAL_B
print(f"Active fraction: {frac:.3%}")
assert abs(frac - 0.0443) < 0.001              # ~4.4%

# compute-vs-memory trade: FP8 checkpoint is ~78 GB (1 byte/param)
mem_fp8_gb = TOTAL_B                            # ~78 GB
print(f"FP8 checkpoint: ~{mem_fp8_gb:.0f} GB — compute like {ACTIVE_B}B, memory like {TOTAL_B}B")
# minimum configs from the release: 2x A100 80GB (160GB) / 2x H100 SXM5 / 1x H200 (141GB) / 1x B200 (192GB)
for name, gb in [("2x A100 80GB", 160), ("1x H200", 141), ("1x B200", 192)]:
    assert gb > mem_fp8_gb + 10, name           # headroom for KV cache
    print(f"{name}: fits (KV-cache headroom {gb - mem_fp8_gb:.0f} GB)")


## 1. Routing load: 6 of 384 experts, 50 layers
Each layer's router forwards each token to the top-6 of 384 routed experts (plus 1 shared expert that always fires). Simulate 1M tokens and check load balance.


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
LAYERS, EXPERTS, TOPK, TOKENS = 50, 384, 6, 1_000_000
# toy router: random top-6 (real routers are learned, but capacity math is the same)
loads = np.zeros((LAYERS, EXPERTS), dtype=np.int64)
for _ in range(TOKENS):
    loads[:, rng.choice(EXPERTS, TOPK, replace=False)] += 1
per_expert = loads.mean()
expected = TOKENS * TOPK / EXPERTS
print(f"Expected hits/expert: {expected:,.0f} | observed mean: {per_expert:,.0f}")
print(f"Min/Max expert load: {loads.min():,} / {loads.max():,}")
assert abs(per_expert - expected) / expected < 0.01
print(f"Active expert slots per token: {TOPK} routed + 1 shared, x {LAYERS} layers = {(TOPK + 1) * LAYERS}")
# expert-selection density
print(f"Share of experts consulted per layer: {TOPK / EXPERTS:.2%}")


## 2. UniBPE fertility: how much more German fits in the window
Independent check (Basic Law, 185 KB legal German): Kolibri 35,190 tokens vs GPT-5 tokenizer (o200k_base) 41,482 tokens. Fewer tokens per word means more text in the same context window — and every token is a forward pass.


In [ ]:
kolibri_tok, o200k_tok = 35190, 41482
saving = 1 - kolibri_tok / o200k_tok
print(f"Token saving on legal German: {saving:.1%}")
assert abs(saving - 0.1518) < 0.002            # ~15.2%

CTX = 262_144                                 # recommended serving window
# How much extra German fits in the same window with the cheaper tokenizer?
extra_tokens = CTX * saving
print(f"Same 262k window fits {extra_tokens:,.0f} more tokens' worth of German vs o200k_base")
print(f"That's ~{extra_tokens / kolibri_tok:.1f}x the full German Basic Law ({kolibri_tok:,} tokens)")
# Aleph Alpha's own figure (11.2% vs best of 10 on German) as a lower bound
saving_aa = 0.112
print(f"Using Aleph Alpha's 11.2% figure: {CTX * saving_aa:,.0f} tokens freed")


## 3. Serving math: from the 170 tok/s RTX Pro 6000 datapoint
A HN commenter hit ~170 tok/s on a single RTX Pro 6000 (FP8). Sanity-check against the 3.46B active parameters, then price a 1M-document workload.


In [ ]:
ACTIVE = 3.46e9
tok_s = 170
# rough dense-transformer estimate: ~2*params FLOPs/token
flops_per_tok = 2 * ACTIVE
achieved_tflops = flops_per_tok * tok_s / 1e12
print(f"~{achieved_tflops:.0f} TFLOP/s implied at {tok_s} tok/s with {ACTIVE/1e9:.2f}B active")
# B200 BF16 dense is ~2250 TFLOP/s; MoE sparse memory traffic is the binding constraint,
# so the number below peak is expected (memory-bound decode)
assert achieved_tflops < 2250

# Workload: translate 1M German tokens (legal docs) — Kolibri vs a same-quality dense 75B
DOC_TOKENS_KOLIBRI = 1_000_000
DOC_TOKENS_DENSE_TOK = int(DOC_TOKENS_KOLIBRI / (1 - 0.1518))  # dense tokenizer needs more tokens
cost_per_mtok = 2.0  # $/Mtok illustrative serving cost
kolibri_cost = DOC_TOKENS_KOLIBRI / 1e6 * cost_per_mtok
dense_tok_cost = DOC_TOKENS_DENSE_TOK / 1e6 * cost_per_mtok
print(f"Kolibri tokens: {DOC_TOKENS_KOLIBRI:,}  -> ${kolibri_cost:.2f}")
print(f"Same text, o200k tokenizer: {DOC_TOKENS_DENSE_TOK:,} -> ${dense_tok_cost:.2f}")
print(f"Tokenizer alone saves ${dense_tok_cost - kolibri_cost:.2f} ({(1 - kolibri_cost/dense_tok_cost):.1%}) on this workload")


## Takeaways
- 78.1B stored / 3.46B awake = 4.4%: the MoE trade is memory-for-compute.
- Random-routing capacity math: each expert sees ~15,625 of 1M tokens — real routers skew this, which is why load-balancing loss terms exist.
- UniBPE's ~15% German token saving compounds: cheaper decode, more German per context window.
- The 1M context is architecturally subsidized (10 full layers + 40 local layers); 262k is the recommended serving point.
